<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import timm
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import os
from PIL import Image

In [19]:
model = timm.create_model('convnext_nano', pretrained=True)
data_config = timm.data.resolve_model_data_config(model)
val_transforms = timm.data.create_transform(**data_config, is_training=False)
train_transforms = timm.data.create_transform(**data_config, is_training=True)

model.safetensors:   0%|          | 0.00/62.4M [00:00<?, ?B/s]

In [20]:
device = 'cuda'
model = model.to(device)

In [26]:
model.head.fc = nn.Linear(in_features=640, out_features=2, bias=True).to(device)

In [7]:
!unzip /content/dog_and_cat_2.zip

Archive:  /content/dog_and_cat_2.zip
  inflating: train_transformed/cat1.jpg  
  inflating: train_transformed/cat10.jpg  
  inflating: train_transformed/cat100.jpg  
  inflating: train_transformed/cat1000.jpg  
  inflating: train_transformed/cat101.jpg  
  inflating: train_transformed/cat102.jpg  
  inflating: train_transformed/cat103.jpg  
  inflating: train_transformed/cat104.jpg  
  inflating: train_transformed/cat105.jpg  
  inflating: train_transformed/cat106.jpg  
  inflating: train_transformed/cat107.jpg  
  inflating: train_transformed/cat108.jpg  
  inflating: train_transformed/cat109.jpg  
  inflating: train_transformed/cat11.jpg  
  inflating: train_transformed/cat110.jpg  
  inflating: train_transformed/cat111.jpg  
  inflating: train_transformed/cat112.jpg  
  inflating: train_transformed/cat113.jpg  
  inflating: train_transformed/cat114.jpg  
  inflating: train_transformed/cat115.jpg  
  inflating: train_transformed/cat116.jpg  
  inflating: train_transformed/cat117.jpg 

In [27]:
class animaldataset(Dataset):
    def __init__(self, transform=None):
        self.data = []
        self.label = []
        self.class2idx = {"cat": 0, "dog": 1}
        self.transform = transform
        for x in os.listdir("/content/train_transformed"):
            self.data.append(os.path.join("/content/train_transformed", x))
            lbl = x[:3]
            self.label.append(self.class2idx[lbl])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, x):
        img = Image.open(self.data[x]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.label[x]

In [28]:
data = animaldataset()
train_size = int(0.8 * len(data))
val_size = len(data) - train_size
train_subset, val_subset = random_split(data, [train_size, val_size])

In [29]:
train_subset.dataset.transform = train_transforms
val_subset.dataset.transform = val_transforms

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

epochs = 5

In [30]:
from tqdm import tqdm
for epoch in range(epochs):
    model.train()
    train_loss, train_correct = 0.0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)

    epoch_train_loss = train_loss / len(train_subset)
    epoch_train_acc = train_correct.double() / len(train_subset)

    model.eval()
    val_loss, val_correct = 0.0, 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            images, labels = images.cuda(), labels.cuda()

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_subset)
    epoch_val_acc = val_correct.double() / len(val_subset)

    print(f"Epoch {epoch+1} -> Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

Epoch 1/5 [Val]: 100%|██████████| 13/13 [00:03<00:00,  3.45it/s]


Epoch 1 -> Train Loss: 0.1369 | Train Acc: 0.9406 | Val Loss: 0.1065 | Val Acc: 0.9650


Epoch 2/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  7.26it/s]


Epoch 2 -> Train Loss: 0.0131 | Train Acc: 0.9969 | Val Loss: 0.0786 | Val Acc: 0.9725


Epoch 3/5 [Val]: 100%|██████████| 13/13 [00:02<00:00,  6.48it/s]


Epoch 3 -> Train Loss: 0.0039 | Train Acc: 0.9988 | Val Loss: 0.1412 | Val Acc: 0.9675


Epoch 4/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  7.60it/s]


Epoch 4 -> Train Loss: 0.0063 | Train Acc: 0.9981 | Val Loss: 0.1062 | Val Acc: 0.9725


Epoch 5/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  7.40it/s]

Epoch 5 -> Train Loss: 0.0036 | Train Acc: 0.9988 | Val Loss: 0.0862 | Val Acc: 0.9750


In [31]:
print(torch.cuda.is_available())

True


In [32]:
state_dict = model.state_dict()
torch.save(state_dict, 'best_model.pth')